# Soup GRPO task: step 2, training dry run on Qwen3-4B (Kaggle T4)

**Before running:** Settings, then **Accelerator: GPU T4 x2** and **Internet: On**. Then **Save Version ->
Save & Run All (Commit)** and close the tab. Takes about **1 h 45 min**. When the version shows Complete, open
its **Output** tab and download **`soup_results.zip`**.

Everything here is pre-registered in `predictions/PREDICTIONS.md` ("Pilot 2 outcome and the training setup"):
1-5. Same safety checks as before (8 GB cap, tests, CUDA equivalence, PCIe).
6. **Memory budget** at the real training size: generate 48 answers x 512 tokens, one training pass, one Adam step.
7. **Resume test on the 4B** (kill -9 mid-step, resume) and an **lr = 0** check.
8. **Dry run: 5 real GRPO steps** with the final configuration (time per step, zero-advantage share, memory).
9. **100 extra pilot puzzles** for the chosen level (5 numbers), as the selection rule requires.

In [ ]:
# 1. Unpack code + data (git c24ffa0+dirty, bundle sha256 fa36a1de5855575a)
import base64, hashlib, io, json, os, subprocess, sys, tarfile, time
from pathlib import Path
LOCAL_TEST = os.environ.get("SOUP_LOCAL_TEST") == "1"   # only for testing this notebook on a laptop
WORK = Path(os.environ.get("SOUP_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = "..."
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == "fa36a1de5855575a7ca31167850b3c8400f95fec058311d20d9b9b333124f11e", "bundle corrupted"
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    tar.extractall(WORK, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
REPO = WORK / "soup-grpo"
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
man = json.loads((REPO / "data/MANIFEST.json").read_text())
for f, info in man["files"].items():
    assert hashlib.sha256((REPO / "data" / f).read_bytes()).hexdigest() == info["sha256"], f
(REPO / "results/logs").mkdir(parents=True, exist_ok=True)
(REPO / "results/pilot").mkdir(parents=True, exist_ok=True)
print("code at", REPO, "| data files verified:", len(man["files"]), "| bundle git rev c24ffa0+dirty")

In [ ]:
# helper: run a command, stream its output live, keep a copy in results/logs
def run(cmd, log=None, check=True, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    t0 = time.time()
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    lines = []
    for line in p.stdout:
        if "Warning" in line or "warn(" in line:
            continue
        print(line, end="", flush=True)
        lines.append(line)
    p.wait()
    if log:
        (REPO / "results/logs" / log).write_text("".join(lines))
    print(f"[exit {p.returncode}, {time.time() - t0:.0f}s]", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return p.returncode

PY = sys.executable
STOP = None   # set by the first failing step; later steps are skipped, results are still packed
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU, as in the task
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less cache fragmentation (pilot 1: 2 GiB)
MODEL = os.environ.get("SOUP_MODEL") or ("Qwen/Qwen3-0.6B" if LOCAL_TEST else "Qwen/Qwen3-4B")   # SOUP_MODEL: local tests only
DEVICE = "cpu" if LOCAL_TEST else "cuda"

In [ ]:
# 2. Environment + dependencies
if not LOCAL_TEST:
    run([PY, "-m", "pip", "install", "-q", "transformers>=4.51", "safetensors>=0.4", "huggingface_hub"], check=False)
    TS = time.strftime("%Y%m%d-%H%M%S")
    env_txt = subprocess.run("nvidia-smi; nvidia-smi --query-gpu=name,memory.total,driver_version,"
                             "pcie.link.gen.current,pcie.link.gen.max,pcie.link.width.current,pcie.link.width.max "
                             "--format=csv; free -g; nproc; df -h /kaggle/working",
                             shell=True, capture_output=True, text=True).stdout
    import torch, transformers
    env_txt += f"\ntorch {torch.__version__} cuda {torch.version.cuda} transformers {transformers.__version__}"
    env_txt += f"\ndevice {torch.cuda.get_device_name(0)} capability {torch.cuda.get_device_capability(0)}"
    env_txt += f" bf16_supported {torch.cuda.is_bf16_supported()}"
    (REPO / f"results/logs/env_{TS}.txt").write_text(env_txt)
    print(env_txt)

In [ ]:
# 3. Session monitor (nvidia-smi + host RAM every second) and the 8 GB ballast
if STOP is None:
    try:
        if not LOCAL_TEST:
            TS = time.strftime("%Y%m%d-%H%M%S")
            smi_log = open(REPO / f"results/logs/nvsmi_{TS}.csv", "w")
            monitor = subprocess.Popen(["nvidia-smi", "--query-gpu=timestamp,memory.used,memory.total,utilization.gpu,"
                                        "pcie.link.gen.current,pcie.link.width.current,temperature.gpu,power.draw,clocks.sm",
                                        "--format=csv", "-l", "1"], stdout=smi_log, stderr=subprocess.STDOUT)
            host_mon = subprocess.Popen(["bash", "-c", f"while true; do echo \"$(date -Is),$(free -m | awk '/Mem:/ {{print $3\",\"$7}}')\""
                                         f" >> {REPO}/results/logs/hostmem_{TS}.csv; sleep 1; done"])
            ballast = subprocess.Popen([PY, "scripts/gpu/ballast.py", "--cap-mib", "8192"],
                                       stdout=open(REPO / "results/logs/ballast.out", "w"), stderr=subprocess.STDOUT)
            for _ in range(120):
                if (REPO / "results/logs/ballast.json").exists():
                    break
                time.sleep(1)
            BALLAST = json.loads((REPO / "results/logs/ballast.json").read_text())
            print("ballast:", BALLAST)
            assert BALLAST["free_left_mib"] <= 8192, "cap not in place"
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 4. Model download (~8 GB) and correctness of the engine on this GPU
if STOP is None:
    try:
        t0 = time.time()
        from soupgrpo.runtime import resolve_model_dir
        print("downloading", MODEL, "(~8 GB, a few minutes; no output until done) ...", flush=True)
        print("model at", resolve_model_dir(MODEL), f"({time.time() - t0:.0f}s)")
        run([PY, "-m", "pytest", "-q", "-x"], log="pytest.txt", check=False)
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/cuda_equivalence.py"], log="cuda_equivalence.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 5. PCIe bandwidth (pinned vs pageable, one 4B layer = 202 MB) + CUDA context cost
if STOP is None:
    try:
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/pcie_bench.py"], log="pcie.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 6. Memory budget at the real training size (H11b, H12, H13): 48 answers x 512 tokens, train pass, Adam step
if STOP is None:
    try:
        if not LOCAL_TEST:
            rc = run([PY, "scripts/gpu/budget_check.py", "--batch", "48", "--max-new", "512", "--micro-batch", "8",
                      "--dtype", "fp16", "--data", "data/train_n5_main.jsonl", "--prompt", "v2"], log="budget.txt", check=False)
            found = sorted((REPO / "results/logs").glob("budget_*.json"))
            if rc != 0 or not found:
                # a crash of this script (not an over-cap result) must not cost the dry run: the trainer logs memory too
                print("WARNING: budget_check did not finish (exit", rc, "); continuing, see logs/budget.txt")
            else:
                B = json.loads(found[-1].read_text())
                print("our peak MiB:", B["our_peak_mib"], "| under cap:", B["under_cap"],
                      "| decode pass s:", round(B["decode_pass_s"], 3))
                assert B["under_cap"], "training-size budget does not fit under the cap: do not start the dry run"
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 7. H19 resume test on the 4B (kill -9 mid-step) + lr = 0 check
if STOP is None:
    try:
        DT = "fp32" if LOCAL_TEST else "fp16"
        run([PY, "scripts/gpu/resume_check.py", "--model", MODEL, "--dtype", DT, "--data", "data/train_n5_main.jsonl"],
            log="resume_check.txt")
        R = json.loads(sorted((REPO / "results/logs").glob("resume_check_*.json"))[-1].read_text())
        print("H19 PASS_resume:", R["PASS_resume"], "| PASS_exact:", R["PASS_exact"])
        run([PY, "scripts/train_grpo.py", "--model", MODEL, "--dtype", DT, "--data", "data/train_n5_main.jsonl",
             "--steps", "2", "--prompts-per-step", "2", "--group", "4", "--max-new", "64", "--gen-batch", "8",
             "--train-seqs-per-pass", "8", "--micro-batch", "4", "--lr", "0", "--log-kl-every", "1",
             "--run-dir", REPO / "runs/lr0"], log="lr0.txt")
        M0 = [json.loads(l) for l in open(REPO / "runs/lr0/metrics.jsonl")]
        lr0 = {"adapter_delta_norm": [m["adapter_delta_norm"] for m in M0], "kl_mean": [m["kl_mean"] for m in M0]}
        lr0["PASS"] = all(v == 0 for v in lr0["adapter_delta_norm"]) and all(not k for k in lr0["kl_mean"])
        (REPO / "results/logs/lr0_check.json").write_text(json.dumps(lr0, indent=2))
        print("lr=0 check:", lr0)
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 8. Dry run: 5 GRPO steps with the final configuration (H1, H25, H33, H11b)
if STOP is None:
    try:
        if LOCAL_TEST:
            cfg = ["--steps", "2", "--prompts-per-step", "2", "--group", "4", "--max-new", "32", "--gen-batch", "8",
                   "--train-seqs-per-pass", "8", "--micro-batch", "4", "--dtype", "fp32"]
        else:
            cfg = ["--steps", "5", "--prompts-per-step", "6", "--group", "8", "--max-new", "512", "--gen-batch", "48",
                   "--train-seqs-per-pass", "48", "--micro-batch", "8", "--dtype", "fp16"]
        run([PY, "scripts/train_grpo.py", "--model", MODEL, "--data", "data/train_n5_main.jsonl", "--prompt", "v2",
             "--lr", "2e-5", "--warmup", "2", "--log-kl-every", "2", "--seed", "1234", *cfg,
             "--run-dir", REPO / "runs/dryrun"], log="dryrun.txt")
        M = [json.loads(l) for l in open(REPO / "runs/dryrun/metrics.jsonl")]
        for m in M:
            t = m["time_s"]
            print(f"step {m['step']}: {t['total']:.0f}s (generate {t['generate'] / t['total']:.0%}) reward {m['reward']:.2f} "
                  f"acc|finished {m['acc_finished']} trunc {m['truncated_share']:.0%} zero-adv {m['zero_adv_group_share']:.0%} "
                  f"adapter {m['adapter_delta_norm']:.2e} kl {m['kl_mean']} smi {m['smi_mib_after_gen']}/{m['smi_mib_after_train']}")
        import shutil
        shutil.copytree(REPO / "runs/dryrun", REPO / "results/dryrun", ignore=shutil.ignore_patterns("ckpt"), dirs_exist_ok=True)
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 9. Pilot rule 4: 100 extra n5 puzzles (taken out of the training data), prompt v2, 512 tokens
if STOP is None:
    try:
        run([PY, "scripts/sample_eval.py", "--model", MODEL, "--device", DEVICE, "--dtype", "fp32" if LOCAL_TEST else "fp16",
             "--prompt", "v2", "--data", "data/pilot_n5_extra.jsonl", "--samples", "4", "--temperature", "1.0",
             "--max-new", "48" if LOCAL_TEST else "512", "--batch", "48", "--out", REPO / "results/pilot/n5_v2_512_extra",
             *(["--limit", "4"] if LOCAL_TEST else [])], log="pilot_n5_extra.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 8. Summary with the pre-registered selection rule + answers to read by hand
if STOP is None:
    try:
        run([PY, "scripts/pilot_summary.py", *sorted(map(str, (REPO / "results/pilot").glob("*.summary.json")))],
            log="pilot_summary.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 9. Peak memory of THIS run (nvidia-smi used minus the ballast) and pack results (always runs)
if not LOCAL_TEST and "smi_log" in globals() and "BALLAST" in globals():
    import csv
    rows = list(csv.reader(open(smi_log.name)))[1:]
    used = [int(r[1].strip().split()[0]) for r in rows if len(r) > 1 and r[1].strip().split()[0].isdigit()]
    # subtract everything that was on the card before our processes started (ballast + driver baseline);
    # pilot 1 subtracted only the ballast and over-reported by the 107 MiB baseline (E12)
    peak = max(used) - BALLAST["smi_used_after_mib"]
    cap = {"smi_peak_total_mib": max(used), "smi_before_us_mib": BALLAST["smi_used_after_mib"],
           "ballast_used_mib": BALLAST["ballast_used_mib"], "our_peak_mib": peak,
           "cap_mib": BALLAST["cap_mib"], "under_cap": peak <= BALLAST["cap_mib"], "samples": len(used)}
    (REPO / "results/logs/cap_summary.json").write_text(json.dumps(cap, indent=2))
    print(cap)
import shutil
(REPO / "results/logs/stop_reason.txt").write_text(str(STOP))
print("STOP reason:", STOP)
zip_base = WORK / "soup_results"
shutil.make_archive(str(zip_base), "zip", REPO / "results")
print("DOWNLOAD:", str(zip_base) + ".zip")